In [ ]:
!pip install -q tiktoken

In [ ]:
from pathlib import Path
import os
import sys
import math
import torch

PROJECT_ROOT = Path(
    "/kaggle/input/datasets/arjunbhattarai123/"
    "gpt-2-llm-from-scratch-training-project-1"
)

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(f"Project root not found: {PROJECT_ROOT}")

START_STEP = 500_000
MAX_STEPS = 800_000
ADDITIONAL_STEPS = MAX_STEPS - START_STEP

if ADDITIONAL_STEPS != 300_000:
    raise ValueError("Expected exactly 300,000 additional steps.")

checkpoint_candidates = [
    Path("/kaggle/working/gpt2_wikitext_500k.pt"),
    PROJECT_ROOT / "gpt2_wikitext_500k.pt",
    PROJECT_ROOT / "checkpoints" / "gpt2_wikitext_500k.pt",
    Path(
        "/kaggle/input/datasets/arjunbhattarai123/"
        "gpt2-wikitext-500k/gpt2_wikitext_500k.pt"
    ),
]

if not any(p.is_file() for p in checkpoint_candidates):
    checkpoint_candidates += list(
        Path("/kaggle/input").rglob("gpt2_wikitext_500k.pt")
    )

CHECKPOINT_PATH = next(
    (p for p in checkpoint_candidates if p.is_file()),
    None
)

if CHECKPOINT_PATH is None:
    raise FileNotFoundError(
        "500K checkpoint not found. Add it as a Kaggle input."
    )

OUTPUT_CHECKPOINT = Path(
    "/kaggle/working/gpt2_wikitext_800k.pt"
)

BEST_CHECKPOINT = Path(
    "/kaggle/working/gpt2_wikitext_800k_best.pt"
)

METRICS_PATH = Path(
    "/kaggle/working/training_metrics_500k_to_800k.csv"
)

RESUME_CHECKPOINT_PATH = Path(
    "/kaggle/working/gpt2_wikitext_500k_fresh_scheduler.pt"
)

checkpoint_meta = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

resume_step = int(checkpoint_meta.get("global_step", -1))

if resume_step != START_STEP:
    raise ValueError(
        f"Expected {START_STEP:,} steps, "
        f"found {resume_step:,}."
    )

if "model_state_dict" not in checkpoint_meta:
    raise KeyError("Model weights missing from checkpoint.")

if "optimizer_state_dict" not in checkpoint_meta:
    raise KeyError("Optimizer state missing from checkpoint.")

saved_optimizer = checkpoint_meta["optimizer_state_dict"]

saved_lrs = [
    float(group["lr"])
    for group in saved_optimizer["param_groups"]
]

print("Checkpoint optimizer LR:", saved_lrs)

if not all(0 < lr < float("inf") for lr in saved_lrs):
    raise ValueError("Invalid optimizer learning rate.")

# Preserve model weights, optimizer moments, RNG states and counters.
# Only clear the old scheduler state to create a fresh 500K -> 800K schedule.
checkpoint_meta["scheduler_state_dict"] = None

torch.save(
    checkpoint_meta,
    RESUME_CHECKPOINT_PATH
)

sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.chdir(PROJECT_ROOT)

device = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)

print("\n" + "=" * 55)
print("GPT-2 TRAINING: 500K -> 800K")
print("=" * 55)

print("Project:", PROJECT_ROOT)
print("Original checkpoint:", CHECKPOINT_PATH)
print("Resume checkpoint:", RESUME_CHECKPOINT_PATH)
print("Starting step:", f"{START_STEP:,}")
print("Target step:", f"{MAX_STEPS:,}")
print("Additional steps:", f"{ADDITIONAL_STEPS:,}")
print("Output checkpoint:", OUTPUT_CHECKPOINT)
print("Best checkpoint:", BEST_CHECKPOINT)
print("Metrics:", METRICS_PATH)
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA capability:", torch.cuda.get_device_capability(0))
else:
    raise RuntimeError(
        "CUDA GPU unavailable. Enable a Kaggle GPU accelerator."
    )

print("=" * 55)
print("INITIALIZATION SUCCESSFUL")
print("=" * 55)

In [ ]:
print("Original checkpoint:", CHECKPOINT_PATH)
print("Fresh-scheduler resume checkpoint:", RESUME_CHECKPOINT_PATH)
print("Output checkpoint:", OUTPUT_CHECKPOINT)

In [ ]:
required_files = [
    PROJECT_ROOT / "src" / "llm" / "gpt" / "model.py",
    PROJECT_ROOT / "src" / "llm" / "training" / "trainer.py",
    PROJECT_ROOT / "src" / "llm" / "training" / "checkpoint.py",
    PROJECT_ROOT / "data" / "wikitext-103" / "train.txt",
    PROJECT_ROOT / "data" / "wikitext-103" / "validation.txt",
    CHECKPOINT_PATH,
    RESUME_CHECKPOINT_PATH,
]

missing = [
    str(p) for p in required_files if not p.exists()
]

if missing:
    raise FileNotFoundError(
        "Missing required files:\n" + "\n".join(missing)
    )

print("All required files found.")

In [ ]:
train_path = PROJECT_ROOT / "data" / "wikitext-103" / "train.txt"
validation_path = PROJECT_ROOT / "data" / "wikitext-103" / "validation.txt"

train_text = train_path.read_text(encoding="utf-8")
validation_text = validation_path.read_text(encoding="utf-8")

print(f"Training characters:   {len(train_text):,}")
print(f"Validation characters: {len(validation_text):,}")

In [ ]:
import tiktoken

from llm.preprocessing.dataset import create_data_loader

tokenizer = tiktoken.get_encoding("gpt2")

# Keep the same data geometry as the previous training run.
BATCH_SIZE = 1
CONTEXT_LENGTH = 1024
NUM_WORKERS = 2

train_loader = create_data_loader(
    text=train_text,
    batch_size=BATCH_SIZE,
    max_length=CONTEXT_LENGTH,
    stride=CONTEXT_LENGTH,
    shuffle=True,
    drop_last=True,
    num_workers=NUM_WORKERS,
)

validation_loader = create_data_loader(
    text=validation_text,
    batch_size=BATCH_SIZE,
    max_length=CONTEXT_LENGTH,
    stride=CONTEXT_LENGTH,
    shuffle=False,
    drop_last=False,
    num_workers=NUM_WORKERS,
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(validation_loader))
print("Batch size:", BATCH_SIZE)
print("Context length:", CONTEXT_LENGTH)

In [ ]:
from llm.gpt.config import GPT_CONFIG_124M
from llm.gpt.model import GPTModel

config = GPT_CONFIG_124M.copy()
config["context_length"] = CONTEXT_LENGTH

model = GPTModel(config).to(device)

# Read the exact optimizer parameter-group structure from the checkpoint.
saved_optimizer = checkpoint_meta["optimizer_state_dict"]

if not saved_optimizer or not saved_optimizer.get("param_groups"):
    raise ValueError(
        "Checkpoint does not contain optimizer parameter groups."
    )

saved_groups = saved_optimizer["param_groups"]

all_params = [
    p for p in model.parameters() if p.requires_grad
]

decay_params = [
    p for p in model.parameters()
    if p.requires_grad and p.ndim >= 2
]

no_decay_params = [
    p for p in model.parameters()
    if p.requires_grad and p.ndim < 2
]

if len(saved_groups) == 1:

    if len(saved_groups[0]["params"]) != len(all_params):
        raise ValueError(
            "Single optimizer group size does not match model parameters."
        )

    optimizer_param_groups = [
        {"params": all_params}
    ]

elif len(saved_groups) == 2:

    if (
        len(saved_groups[0]["params"]) != len(decay_params)
        or len(saved_groups[1]["params"]) != len(no_decay_params)
    ):
        raise ValueError(
            "Two-group checkpoint does not match expected "
            "decay/no-decay parameter grouping."
        )

    optimizer_param_groups = [
        {"params": decay_params},
        {"params": no_decay_params},
    ]

else:
    raise ValueError(
        f"Unsupported optimizer group count: {len(saved_groups)}"
    )

first_group = saved_groups[0]

optimizer = torch.optim.AdamW(
    optimizer_param_groups,
    lr=float(first_group["lr"]),
    betas=tuple(first_group.get("betas", (0.9, 0.999))),
    eps=float(first_group.get("eps", 1e-8)),
    weight_decay=float(first_group.get("weight_decay", 0.01)),
)

# Preserve checkpoint weight-decay configuration.
for new_group, old_group in zip(optimizer.param_groups, saved_groups):
    new_group["weight_decay"] = float(
        old_group.get("weight_decay", 0.0)
    )

# Fresh learning-rate schedule for the remaining 300K steps.
MIN_LR = 1e-7
BASE_LR = float(first_group["lr"])

resume_step = int(checkpoint_meta["global_step"])
remaining_steps = MAX_STEPS - resume_step

if remaining_steps <= 0:
    raise ValueError("Target step must exceed resume step.")

if BASE_LR <= MIN_LR:
    raise ValueError(
        f"BASE_LR ({BASE_LR}) must be greater than MIN_LR ({MIN_LR})."
    )

def lr_lambda(step):
    progress = min(
        max(step / remaining_steps, 0.0),
        1.0
    )

    cosine = 0.5 * (
        1.0 + math.cos(math.pi * progress)
    )

    lr = MIN_LR + (BASE_LR - MIN_LR) * cosine

    return lr / BASE_LR


scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda=lr_lambda
)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print("Optimizer groups:", len(saved_groups))
print(
    "Parameters per group:",
    [len(group["params"]) for group in saved_groups]
)
print(
    "Saved optimizer LR:",
    [group["lr"] for group in saved_groups]
)
print(
    "Saved weight decay:",
    [group.get("weight_decay") for group in saved_groups]
)
print(f"Additional optimizer steps: {remaining_steps:,}")
print(
    "Fresh scheduler initial LR:",
    [group["lr"] for group in optimizer.param_groups]
)

In [ ]:
from llm.training.trainer import train_model_simple
from llm.training.checkpoint import load_checkpoint

print("Training function imported successfully.")

In [ ]:

preflight = load_checkpoint(
    model=model,
    optimizer=optimizer,
    scaler=None,
    path=str(RESUME_CHECKPOINT_PATH),
    device=device,
    scheduler=None,
    restore_rng=False,
)

assert int(preflight["global_step"]) == START_STEP

assert preflight.get("scheduler_state_dict") is None

print("Checkpoint/model/optimizer compatibility check passed.")

print(
    "Restored optimizer LR:",
    [group["lr"] for group in optimizer.param_groups]
)

print(
    "Fresh scheduler initial LR:",
    [group["lr"] for group in optimizer.param_groups]
)

print("Old scheduler state will not be restored.")

In [ ]:
EVAL_FREQ = 2_500
EVAL_ITER = 20

print("=" * 60)
print("STARTING GPT-2 TRAINING")
print("=" * 60)

print("Starting global step:", f"{START_STEP:,}")
print("Target global step:", f"{MAX_STEPS:,}")
print("Additional steps:", f"{ADDITIONAL_STEPS:,}")
print("Base learning rate:", BASE_LR)
print("Minimum learning rate:", MIN_LR)
print("Evaluation frequency:", EVAL_FREQ)
print("Evaluation iterations:", EVAL_ITER)
print("Periodic checkpoint interval:", "5,000 steps")

train_losses, validation_losses, tokens_seen = train_model_simple(
    model=model,
    train_loader=train_loader,
    val_loader=validation_loader,
    optimizer=optimizer,
    device=device,
    num_epochs=100,
    eval_freq=EVAL_FREQ,
    eval_iter=EVAL_ITER,
    start_context="The history of the world",
    tokenizer=tokenizer,
    checkpoint_path=str(OUTPUT_CHECKPOINT),
    config=config,
    max_steps=MAX_STEPS,
    resume_from_checkpoint=str(RESUME_CHECKPOINT_PATH),
    scheduler=scheduler,
    gradient_accumulation_steps=1,
    metrics_path=str(METRICS_PATH),
    periodic_checkpoint_steps=5_000,
    best_checkpoint_path=str(BEST_CHECKPOINT),
    eval_generation=False,
)

print("Training function returned.")

In [ ]:
if not OUTPUT_CHECKPOINT.exists():
    raise FileNotFoundError(
        f"Expected output checkpoint was not created: {OUTPUT_CHECKPOINT}"
    )

final_checkpoint = torch.load(
    OUTPUT_CHECKPOINT,
    map_location="cpu",
    weights_only=False
)

final_step = int(final_checkpoint["global_step"])

print("=" * 60)
print("FINAL TRAINING REPORT")
print("=" * 60)

print("Output checkpoint:", OUTPUT_CHECKPOINT)
print("Final global step:", f"{final_step:,}")
print("Final epoch:", final_checkpoint.get("epoch"))
print("Train loss:", final_checkpoint.get("train_loss"))
print("Validation loss:", final_checkpoint.get("val_loss"))
print("Best validation loss:", final_checkpoint.get("best_val_loss"))

print(
    "Optimizer LR:",
    [
        group["lr"]
        for group in final_checkpoint["optimizer_state_dict"]["param_groups"]
    ]
)

print(
    "Scheduler state saved:",
    final_checkpoint.get("scheduler_state_dict") is not None
)

print(
    "Metrics CSV:",
    METRICS_PATH,
    "| Exists:",
    METRICS_PATH.exists()
)

if final_step != MAX_STEPS:
    print(
        f"WARNING: Expected {MAX_STEPS:,}, "
        f"but checkpoint is at {final_step:,}."
    )
else:
    print("SUCCESS: Target 800K step reached.")